# 📊 Sparse Retrieval: BM25 & TF-IDF - The Keyword Masters

## What is Sparse Retrieval?

**Sparse retrieval uses term frequency statistics to match documents based on keyword overlap.**

### The Classic Approach:

```
Document: "The cat sat on the mat"
    ↓
Sparse Vector: [0, 1, 2, 1, 0, 0, 1, 1, 0, ...] (vocab size)
                   ^  ^  ^  ^
                   cat sat on mat
```

**Most values are 0 (sparse!) - only non-zero for words in the document.**

![Sparse vs Dense Vectors](./images/sparse_vs_dense.png)
*Figure 1: Sparse vectors (mostly zeros) vs Dense vectors (all values non-zero)*

## Why Sparse Retrieval Still Matters 🎯

### Advantages:
- ✅ **Fast** - No neural networks needed
- ✅ **Interpretable** - Can see which terms matched
- ✅ **Exact matching** - Great for specific keywords
- ✅ **No training** - Works out of the box
- ✅ **Low resource** - Runs on CPU easily

### When to Use:
- Exact term matching (product names, IDs)
- Code search (function names, variables)
- Legal/medical (specific terminology)
- Combined with dense (hybrid!)

## Classic Algorithms

### 1. **TF-IDF** (Term Frequency - Inverse Document Frequency)
```
TF: How often term appears in document
IDF: How rare the term is across all documents
Score = TF × IDF
```

### 2. **BM25** (Best Match 25) - The Gold Standard ⭐
```
Improved TF-IDF with:
- Saturation (diminishing returns for term frequency)
- Document length normalization
- Tunable parameters (k1, b)
```

**BM25 is the industry standard for sparse retrieval!**

## 1. TF-IDF: The Foundation

In [1]:
# Install required packages
# !pip install scikit-learn rank-bm25

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

# Sample documents
documents = [
    "The cat sat on the mat",
    "The dog sat on the log",
    "Cats and dogs are great pets",
    "Python is a programming language",
    "Machine learning uses Python programming",
]

# Create TF-IDF vectorizer
vectorizer = TfidfVectorizer()
tfidf_matrix = vectorizer.fit_transform(documents)

print("TF-IDF Matrix Shape:", tfidf_matrix.shape)
print(f"Vocabulary size: {len(vectorizer.vocabulary_)}")
print(f"\nVocabulary: {sorted(vectorizer.vocabulary_.keys())}")

TF-IDF Matrix Shape: (5, 20)
Vocabulary size: 20

Vocabulary: ['and', 'are', 'cat', 'cats', 'dog', 'dogs', 'great', 'is', 'language', 'learning', 'log', 'machine', 'mat', 'on', 'pets', 'programming', 'python', 'sat', 'the', 'uses']


In [2]:
# Examine TF-IDF scores
feature_names = vectorizer.get_feature_names_out()

# Show TF-IDF for first document
doc_idx = 0
doc_vector = tfidf_matrix[doc_idx].toarray()[0]

print(f"Document: '{documents[doc_idx]}'\n")
print(f"{'Term':<15} {'TF-IDF Score'}")
print("="*35)

# Get non-zero scores
for idx, score in enumerate(doc_vector):
    if score > 0:
        print(f"{feature_names[idx]:<15} {score:.4f}")

print("\n💡 Common words (the, on) have lower scores than unique words (cat, mat)")

Document: 'The cat sat on the mat'

Term            TF-IDF Score
cat             0.4115
mat             0.4115
on              0.3320
sat             0.3320
the             0.6640

💡 Common words (the, on) have lower scores than unique words (cat, mat)


## 2. TF-IDF Retrieval

In [3]:
class TFIDFRetriever:
    def __init__(self):
        self.vectorizer = TfidfVectorizer()
        self.documents = []
        self.tfidf_matrix = None
        
    def index(self, documents):
        """Index documents using TF-IDF"""
        self.documents = documents
        self.tfidf_matrix = self.vectorizer.fit_transform(documents)
        print(f"✅ Indexed {len(documents)} documents")
        print(f"   Vocabulary size: {len(self.vectorizer.vocabulary_)}")
        
    def search(self, query, top_k=3):
        """Search using TF-IDF cosine similarity"""
        # Transform query
        query_vector = self.vectorizer.transform([query])
        
        # Calculate cosine similarity
        scores = cosine_similarity(query_vector, self.tfidf_matrix)[0]
        
        # Get top-k
        top_indices = np.argsort(scores)[::-1][:top_k]
        
        results = []
        for rank, idx in enumerate(top_indices):
            results.append({
                'rank': rank + 1,
                'score': float(scores[idx]),
                'document': self.documents[idx]
            })
        
        return results

# Test TF-IDF retriever
tfidf_retriever = TFIDFRetriever()
tfidf_retriever.index(documents)

query = "programming with Python"
results = tfidf_retriever.search(query, top_k=3)

print(f"\nQuery: '{query}'\n")
print("Results:")
for r in results:
    print(f"  {r['rank']}. [{r['score']:.4f}] {r['document']}")

✅ Indexed 5 documents
   Vocabulary size: 20

Query: 'programming with Python'

Results:
  1. [0.6279] Python is a programming language
  2. [0.5501] Machine learning uses Python programming
  3. [0.0000] Cats and dogs are great pets


## 3. BM25: The Superior Algorithm

![BM25 Formula](./images/bm25_formula.png)
*Figure 2: BM25 scoring formula with key parameters k1 (term saturation) and b (length normalization)*

In [4]:
from rank_bm25 import BM25Okapi
import nltk
from nltk.tokenize import word_tokenize

# Download tokenizer
nltk.download('punkt', quiet=True)

class BM25Retriever:
    def __init__(self, k1=1.5, b=0.75):
        """
        BM25 Retriever
        
        Parameters:
        k1: Term frequency saturation (1.2-2.0, default 1.5)
        b: Length normalization (0-1, default 0.75)
        """
        self.k1 = k1
        self.b = b
        self.documents = []
        self.bm25 = None
        
    def index(self, documents):
        """Index documents using BM25"""
        self.documents = documents
        
        # Tokenize documents
        tokenized_docs = [word_tokenize(doc.lower()) for doc in documents]
        
        # Create BM25 index
        self.bm25 = BM25Okapi(tokenized_docs, k1=self.k1, b=self.b)
        
        print(f"✅ Indexed {len(documents)} documents with BM25")
        print(f"   Parameters: k1={self.k1}, b={self.b}")
        
    def search(self, query, top_k=3):
        """Search using BM25 scoring"""
        # Tokenize query
        tokenized_query = word_tokenize(query.lower())
        
        # Get BM25 scores
        scores = self.bm25.get_scores(tokenized_query)
        
        # Get top-k
        top_indices = np.argsort(scores)[::-1][:top_k]
        
        results = []
        for rank, idx in enumerate(top_indices):
            results.append({
                'rank': rank + 1,
                'score': float(scores[idx]),
                'document': self.documents[idx]
            })
        
        return results

# Test BM25
bm25_retriever = BM25Retriever()
bm25_retriever.index(documents)

query = "programming with Python"
results = bm25_retriever.search(query, top_k=3)

print(f"\nQuery: '{query}'\n")
print("BM25 Results:")
for r in results:
    print(f"  {r['rank']}. [{r['score']:.4f}] {r['document']}")

✅ Indexed 5 documents with BM25
   Parameters: k1=1.5, b=0.75

Query: 'programming with Python'

BM25 Results:
  1. [0.7070] Machine learning uses Python programming
  2. [0.7070] Python is a programming language
  3. [0.0000] Cats and dogs are great pets


## 4. BM25 vs TF-IDF Comparison

In [5]:
# Compare on same query
test_queries = [
    "cat and dog pets",
    "Python programming language",
    "machine learning",
]

print("TF-IDF vs BM25 Comparison:\n")
print("="*90)

for query in test_queries:
    print(f"\nQuery: '{query}'\n")
    
    # TF-IDF
    tfidf_results = tfidf_retriever.search(query, top_k=1)
    print(f"TF-IDF Top Result:")
    print(f"  [{tfidf_results[0]['score']:.4f}] {tfidf_results[0]['document']}")
    
    # BM25
    bm25_results = bm25_retriever.search(query, top_k=1)
    print(f"\nBM25 Top Result:")
    print(f"  [{bm25_results[0]['score']:.4f}] {bm25_results[0]['document']}")
    print()

print("\n💡 BM25 typically provides better ranking than TF-IDF!")

TF-IDF vs BM25 Comparison:


Query: 'cat and dog pets'

TF-IDF Top Result:
  [0.4082] Cats and dogs are great pets

BM25 Top Result:
  [2.1288] Cats and dogs are great pets


Query: 'Python programming language'

TF-IDF Top Result:
  [0.8349] Python is a programming language

BM25 Top Result:
  [1.8613] Python is a programming language


Query: 'machine learning'

TF-IDF Top Result:
  [0.6818] Machine learning uses Python programming

BM25 Top Result:
  [2.3085] Machine learning uses Python programming


💡 BM25 typically provides better ranking than TF-IDF!


## 5. Understanding BM25 Parameters

In [6]:
# Test different k1 values (term frequency saturation)
k1_values = [0.5, 1.2, 1.5, 2.0, 3.0]

# Long document with repeated terms
test_docs = [
    "Python Python Python programming",  # Lots of repetition
    "Python is a programming language",   # Normal
]

query = "Python programming"

print("Effect of k1 Parameter (Term Frequency Saturation):\n")
print(f"Query: '{query}'\n")
print(f"{'k1':<8} {'Doc 1 (repeated)':<20} {'Doc 2 (normal)'}")
print("="*50)

for k1 in k1_values:
    retriever = BM25Retriever(k1=k1)
    retriever.index(test_docs)
    results = retriever.search(query, top_k=2)
    
    scores = [r['score'] for r in results]
    print(f"{k1:<8} {scores[0]:<20.4f} {scores[1]:.4f}")

print("\n💡 Lower k1 → Less benefit from term repetition")
print("   Higher k1 → More benefit from term repetition")
print("   Default k1=1.5 is usually optimal")

Effect of k1 Parameter (Term Frequency Saturation):

Query: 'Python programming'

k1       Doc 1 (repeated)     Doc 2 (normal)
✅ Indexed 2 documents with BM25
   Parameters: k1=0.5, b=0.75
0.5      -0.3132              -0.3750
✅ Indexed 2 documents with BM25
   Parameters: k1=1.2, b=0.75
1.2      -0.3079              -0.4277
✅ Indexed 2 documents with BM25
   Parameters: k1=1.5, b=0.75
1.5      -0.3066              -0.4453
✅ Indexed 2 documents with BM25
   Parameters: k1=2.0, b=0.75
2.0      -0.3049              -0.4701
✅ Indexed 2 documents with BM25
   Parameters: k1=3.0, b=0.75
3.0      -0.3030              -0.5076

💡 Lower k1 → Less benefit from term repetition
   Higher k1 → More benefit from term repetition
   Default k1=1.5 is usually optimal


In [7]:
# Test different b values (document length normalization)
b_values = [0.0, 0.25, 0.5, 0.75, 1.0]

test_docs_length = [
    "Python",  # Very short
    "Python is a high-level programming language used for web development and data science",  # Long
]

query = "Python"

print("\nEffect of b Parameter (Length Normalization):\n")
print(f"Query: '{query}'\n")
print(f"{'b':<8} {'Short Doc':<20} {'Long Doc'}")
print("="*50)

for b in b_values:
    retriever = BM25Retriever(k1=1.5, b=b)
    retriever.index(test_docs_length)
    results = retriever.search(query, top_k=2)
    
    scores = [r['score'] for r in results]
    print(f"{b:<8} {scores[0]:<20.4f} {scores[1]:.4f}")

print("\n💡 b=0 → No length normalization (longer docs favored)")
print("   b=1 → Full normalization (penalizes long docs)")
print("   Default b=0.75 balances both")


Effect of b Parameter (Length Normalization):

Query: 'Python'

b        Short Doc            Long Doc
✅ Indexed 2 documents with BM25
   Parameters: k1=1.5, b=0.0
0.0      -0.0310              -0.0310
✅ Indexed 2 documents with BM25
   Parameters: k1=1.5, b=0.25
0.25     -0.0274              -0.0355
✅ Indexed 2 documents with BM25
   Parameters: k1=1.5, b=0.5
0.5      -0.0246              -0.0417
✅ Indexed 2 documents with BM25
   Parameters: k1=1.5, b=0.75
0.75     -0.0223              -0.0504
✅ Indexed 2 documents with BM25
   Parameters: k1=1.5, b=1.0
1.0      -0.0204              -0.0637

💡 b=0 → No length normalization (longer docs favored)
   b=1 → Full normalization (penalizes long docs)
   Default b=0.75 balances both


## 6. Advanced: Custom Text Processing

In [8]:
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
import string

nltk.download('stopwords', quiet=True)

class AdvancedBM25Retriever:
    def __init__(self, remove_stopwords=True, use_stemming=True, k1=1.5, b=0.75):
        self.remove_stopwords = remove_stopwords
        self.use_stemming = use_stemming
        self.k1 = k1
        self.b = b
        
        self.stopwords = set(stopwords.words('english')) if remove_stopwords else set()
        self.stemmer = PorterStemmer() if use_stemming else None
        
        self.documents = []
        self.bm25 = None
        
    def preprocess(self, text):
        """Advanced preprocessing"""
        # Lowercase
        text = text.lower()
        
        # Remove punctuation
        text = text.translate(str.maketrans('', '', string.punctuation))
        
        # Tokenize
        tokens = text.split()
        
        # Remove stopwords
        if self.remove_stopwords:
            tokens = [t for t in tokens if t not in self.stopwords]
        
        # Stemming
        if self.use_stemming:
            tokens = [self.stemmer.stem(t) for t in tokens]
        
        return tokens
    
    def index(self, documents):
        self.documents = documents
        tokenized_docs = [self.preprocess(doc) for doc in documents]
        self.bm25 = BM25Okapi(tokenized_docs, k1=self.k1, b=self.b)
        print(f"✅ Indexed with preprocessing (stopwords={self.remove_stopwords}, stemming={self.use_stemming})")
    
    def search(self, query, top_k=3):
        tokenized_query = self.preprocess(query)
        scores = self.bm25.get_scores(tokenized_query)
        top_indices = np.argsort(scores)[::-1][:top_k]
        
        return [{
            'rank': rank + 1,
            'score': float(scores[idx]),
            'document': self.documents[idx]
        } for rank, idx in enumerate(top_indices)]

# Compare configurations
configs = [
    ("Basic", False, False),
    ("+ Stopwords", True, False),
    ("+ Stemming", False, True),
    ("Full", True, True),
]

query = "running and jumping exercises"

docs_for_preprocessing = [
    "The runners are running in the marathon",
    "Jumping exercises improve athletic performance",
    "Python programming for data science",
]

print(f"Query: '{query}'\n")
print(f"{'Configuration':<20} {'Top Match'}")
print("="*80)

for name, stop, stem in configs:
    retriever = AdvancedBM25Retriever(remove_stopwords=stop, use_stemming=stem)
    retriever.index(docs_for_preprocessing)
    results = retriever.search(query, top_k=1)
    print(f"{name:<20} [{results[0]['score']:.4f}] {results[0]['document']}")

print("\n💡 Stemming helps match 'running' with 'runners'!")

Query: 'running and jumping exercises'

Configuration        Top Match
✅ Indexed with preprocessing (stopwords=False, stemming=False)
Basic                [1.0788] Jumping exercises improve athletic performance
✅ Indexed with preprocessing (stopwords=True, stemming=False)
+ Stopwords          [0.9183] Jumping exercises improve athletic performance
✅ Indexed with preprocessing (stopwords=False, stemming=True)
+ Stemming           [1.0788] Jumping exercises improve athletic performance
✅ Indexed with preprocessing (stopwords=True, stemming=True)
Full                 [0.9183] Jumping exercises improve athletic performance

💡 Stemming helps match 'running' with 'runners'!


## 7. Production-Ready BM25 System

In [9]:
import pickle
from typing import List, Dict, Optional

class ProductionBM25:
    def __init__(self, k1=1.5, b=0.75, preprocess=True):
        self.k1 = k1
        self.b = b
        self.preprocess_enabled = preprocess
        
        self.documents = []
        self.metadata = []
        self.bm25 = None
        
        if preprocess:
            self.stopwords = set(stopwords.words('english'))
            self.stemmer = PorterStemmer()
    
    def preprocess(self, text: str) -> List[str]:
        if not self.preprocess_enabled:
            return word_tokenize(text.lower())
        
        text = text.lower().translate(str.maketrans('', '', string.punctuation))
        tokens = text.split()
        tokens = [t for t in tokens if t not in self.stopwords]
        tokens = [self.stemmer.stem(t) for t in tokens]
        return tokens
    
    def index(self, documents: List[str], metadata: Optional[List[Dict]] = None):
        """Index documents with optional metadata"""
        self.documents = documents
        self.metadata = metadata if metadata else [{} for _ in documents]
        
        tokenized = [self.preprocess(doc) for doc in documents]
        self.bm25 = BM25Okapi(tokenized, k1=self.k1, b=self.b)
        
        print(f"✅ Indexed {len(documents)} documents")
    
    def search(self, query: str, top_k: int = 5, score_threshold: float = 0.0):
        """Search with score filtering"""
        tokenized_query = self.preprocess(query)
        scores = self.bm25.get_scores(tokenized_query)
        
        # Filter by threshold
        valid_indices = np.where(scores >= score_threshold)[0]
        valid_scores = scores[valid_indices]
        
        # Get top-k from valid
        top_k = min(top_k, len(valid_scores))
        top_positions = np.argsort(valid_scores)[::-1][:top_k]
        top_indices = valid_indices[top_positions]
        
        return [{
            'rank': rank + 1,
            'score': float(scores[idx]),
            'document': self.documents[idx],
            'metadata': self.metadata[idx],
            'index': int(idx)
        } for rank, idx in enumerate(top_indices)]
    
    def save(self, path: str):
        """Save index to disk"""
        with open(path, 'wb') as f:
            pickle.dump({
                'documents': self.documents,
                'metadata': self.metadata,
                'bm25': self.bm25,
                'config': {'k1': self.k1, 'b': self.b, 'preprocess': self.preprocess_enabled}
            }, f)
        print(f"✅ Saved index to {path}")
    
    def load(self, path: str):
        """Load index from disk"""
        with open(path, 'rb') as f:
            data = pickle.load(f)
        
        self.documents = data['documents']
        self.metadata = data['metadata']
        self.bm25 = data['bm25']
        
        config = data['config']
        self.k1 = config['k1']
        self.b = config['b']
        self.preprocess_enabled = config['preprocess']
        
        print(f"✅ Loaded index from {path}")

# Test production system
prod_bm25 = ProductionBM25()

docs_with_meta = [
    ("Machine learning is transforming industries", {"category": "AI", "date": "2024-01"}),
    ("Python is great for data science", {"category": "Programming", "date": "2024-02"}),
    ("RAG improves language model accuracy", {"category": "AI", "date": "2024-03"}),
]

docs = [d[0] for d in docs_with_meta]
meta = [d[1] for d in docs_with_meta]

prod_bm25.index(docs, metadata=meta)

results = prod_bm25.search("artificial intelligence and machine learning", top_k=2)

print("\nSearch Results:")
for r in results:
    print(f"\n{r['rank']}. {r['document']}")
    print(f"   Score: {r['score']:.4f}")
    print(f"   Metadata: {r['metadata']}")

✅ Indexed 3 documents

Search Results:

1. Machine learning is transforming industries
   Score: 1.0583
   Metadata: {'category': 'AI', 'date': '2024-01'}

2. RAG improves language model accuracy
   Score: 0.0000
   Metadata: {'category': 'AI', 'date': '2024-03'}


## 8. Real-World RAG Example

In [10]:
# Realistic RAG knowledge base
rag_knowledge = [
    "RAG (Retrieval-Augmented Generation) combines information retrieval with language model generation.",
    "BM25 is a popular ranking function used in information retrieval and search engines.",
    "Sparse retrieval methods like BM25 excel at exact keyword matching and specific term queries.",
    "Dense retrieval uses neural embeddings to capture semantic similarity between texts.",
    "Hybrid retrieval systems combine sparse (BM25) and dense methods for optimal performance.",
    "Vector databases store and efficiently query high-dimensional embeddings for RAG systems.",
    "The k1 parameter in BM25 controls term frequency saturation, typically set between 1.2 and 2.0.",
    "Document length normalization in BM25 is controlled by the b parameter, usually around 0.75.",
    "TF-IDF is a simpler alternative to BM25 but lacks sophisticated length normalization.",
    "Modern RAG systems often use BM25 as a first-stage retriever before neural re-ranking.",
]

# Create retriever
rag_bm25 = ProductionBM25(k1=1.5, b=0.75)
rag_bm25.index(rag_knowledge)

# User questions
questions = [
    "What is BM25 and how does it work?",
    "How to tune BM25 parameters?",
    "Difference between sparse and dense retrieval?",
    "What is hybrid retrieval?",
]

print("RAG Knowledge Base Q&A (BM25 Retrieval):\n")
print("="*80)

for question in questions:
    results = rag_bm25.search(question, top_k=2)
    
    print(f"\n❓ {question}")
    print(f"\n📄 Retrieved Context:")
    for r in results:
        print(f"   [{r['score']:.2f}] {r['document']}")

print("\n💡 These would be passed to an LLM for answer generation!")

✅ Indexed 10 documents
RAG Knowledge Base Q&A (BM25 Retrieval):


❓ What is BM25 and how does it work?

📄 Retrieved Context:
   [0.43] TF-IDF is a simpler alternative to BM25 but lacks sophisticated length normalization.
   [0.41] BM25 is a popular ranking function used in information retrieval and search engines.

❓ How to tune BM25 parameters?

📄 Retrieved Context:
   [1.60] Document length normalization in BM25 is controlled by the b parameter, usually around 0.75.
   [1.53] The k1 parameter in BM25 controls term frequency saturation, typically set between 1.2 and 2.0.

❓ Difference between sparse and dense retrieval?

📄 Retrieved Context:
   [2.81] Hybrid retrieval systems combine sparse (BM25) and dense methods for optimal performance.
   [1.68] Dense retrieval uses neural embeddings to capture semantic similarity between texts.

❓ What is hybrid retrieval?

📄 Retrieved Context:
   [2.22] Hybrid retrieval systems combine sparse (BM25) and dense methods for optimal performance.
   

## 9. Sparse vs Dense: When to Use What

In [11]:
# Comparison scenarios
from sentence_transformers import SentenceTransformer

# Load dense model for comparison
dense_model = SentenceTransformer('all-MiniLM-L6-v2')

# Test cases
test_cases = [
    {
        'name': 'Exact Match',
        'query': 'iPhone 15 Pro Max',
        'docs': [
            'iPhone 15 Pro Max specifications and pricing',
            'The latest Apple smartphone with great features',
            'Samsung Galaxy S24 Ultra review',
        ]
    },
    {
        'name': 'Semantic Match',
        'query': 'canine companions',
        'docs': [
            'Dogs are loyal pets and great friends',
            'Cats are independent animals',
            'Canine training and behavior guide',
        ]
    },
    {
        'name': 'Synonyms',
        'query': 'automobile repair',
        'docs': [
            'Car maintenance and repair services',
            'Vehicle servicing and inspection',
            'Bicycle repair shop locations',
        ]
    },
]

print("Sparse (BM25) vs Dense (Semantic) Comparison:\n")
print("="*90)

for case in test_cases:
    print(f"\n{case['name']}: '{case['query']}'\n")
    
    # BM25
    bm25_temp = ProductionBM25()
    bm25_temp.index(case['docs'])
    bm25_result = bm25_temp.search(case['query'], top_k=1)[0]
    
    # Dense
    query_emb = dense_model.encode(case['query'])
    doc_embs = dense_model.encode(case['docs'])
    dense_scores = cosine_similarity([query_emb], doc_embs)[0]
    dense_idx = np.argmax(dense_scores)
    
    print(f"BM25: {bm25_result['document']}")
    print(f"Dense: {case['docs'][dense_idx]}")
    
    if bm25_result['document'] == case['docs'][dense_idx]:
        print("✅ Both methods agree")
    else:
        print("⚠️ Methods disagree")

print("\n💡 BM25 wins for exact matches, Dense wins for semantic understanding!")

Sparse (BM25) vs Dense (Semantic) Comparison:


Exact Match: 'iPhone 15 Pro Max'

✅ Indexed 3 documents
BM25: iPhone 15 Pro Max specifications and pricing
Dense: iPhone 15 Pro Max specifications and pricing
✅ Both methods agree

Semantic Match: 'canine companions'

✅ Indexed 3 documents
BM25: Canine training and behavior guide
Dense: Dogs are loyal pets and great friends
⚠️ Methods disagree

Synonyms: 'automobile repair'

✅ Indexed 3 documents
BM25: Bicycle repair shop locations
Dense: Car maintenance and repair services
⚠️ Methods disagree

💡 BM25 wins for exact matches, Dense wins for semantic understanding!


## Key Takeaways 🎯

### ✅ Sparse Retrieval Strengths:

1. **Fast** - No GPU needed, runs on CPU
2. **Interpretable** - See exactly which terms matched
3. **Exact matching** - Perfect for specific keywords
4. **Zero-shot** - No training required
5. **Deterministic** - Same query = same results

### 🔑 BM25 Formula (Simplified):

```
BM25(Q,D) = Σ IDF(qi) × [f(qi,D) × (k1 + 1)] / [f(qi,D) + k1 × (1 - b + b × |D|/avgdl)]

Where:
- IDF(qi) = Inverse document frequency of term qi
- f(qi,D) = Frequency of qi in document D
- k1 = Term saturation parameter (default 1.5)
- b = Length normalization (default 0.75)
- |D| = Document length
- avgdl = Average document length
```

### 📊 Parameter Tuning Guide:

| Parameter | Range | Effect | When to Adjust |
|-----------|-------|--------|----------------|
| **k1** | 1.2-2.0 | Term frequency saturation | Higher for verbose docs |
| **b** | 0.0-1.0 | Length normalization | Lower for varying lengths |

**Defaults (k1=1.5, b=0.75) work well for most cases!**

### 🚀 When to Use BM25:

**Perfect for:**
- ✅ Exact term matching (product codes, names)
- ✅ Technical documentation (APIs, code)
- ✅ Legal/medical text (specific terminology)
- ✅ First-stage retrieval (before re-ranking)
- ✅ Hybrid systems (with dense retrieval)

**Not ideal for:**
- ❌ Semantic similarity (use dense)
- ❌ Handling synonyms (use dense)
- ❌ Multilingual search (use dense)
- ❌ Conceptual queries (use dense)

### 💡 Production Tips:

```python
# 1. Preprocessing matters
- Remove stopwords for cleaner matching
- Use stemming for morphological variants
- Lowercase everything

# 2. Tune for your domain
- Short docs (tweets): k1=1.2, b=0.5
- Long docs (articles): k1=1.8, b=0.9
- Mixed: k1=1.5, b=0.75 (default)

# 3. Optimize storage
- Save tokenized docs (avoid re-tokenizing)
- Use pickle for fast loading
- Cache common queries

# 4. Combine with dense
- BM25 for recall (cast wide net)
- Dense for precision (semantic re-ranking)
```

### 📈 Performance Benchmarks:

```
BM25 Speed (1M documents):
- Indexing: ~1-2 minutes (CPU)
- Query: ~10-50ms per query
- Memory: ~500MB-1GB

vs Dense Retrieval:
- Indexing: ~10-30 minutes (GPU)
- Query: ~100-500ms per query  
- Memory: ~5-10GB
```

### 🎯 Decision Matrix:

| Use Case | Method | Reason |
|----------|--------|--------|
| Product search | BM25 | Exact names/codes |
| FAQ matching | Dense | Semantic similarity |
| Code search | BM25 | Function/variable names |
| Question answering | Hybrid | Best of both |
| Document search | Hybrid | Comprehensive retrieval |

## Next Steps 🔜

Move to `03_Hybrid_Retrieval.ipynb` to combine sparse and dense!

That's where we get the BEST of both worlds! 🚀